<a href="https://colab.research.google.com/github/Muhammad-Rameez06/rag-experiments/blob/main/RAG_EXPERIMENTS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sentence_transformers import SentenceTransformer, util
import torch

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-V2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from sentence_transformers import util

documents = [
    "I love cats.",
    "I like kittens.",
    "Dogs are friendly animals.",
    "The stock market is falling.",
    "Python is a programming language."
]

doc_embeddings = model.encode(documents)

# Full pairwise cosine similarity matrix
similarity_matrix = util.cos_sim(doc_embeddings, doc_embeddings)

# Print every unique pair (i, j) with i < j
for i in range(len(documents)):
    for j in range(i + 1, len(documents)):
        score = similarity_matrix[i][j].item()
        print(f"Similarity({documents[i]!r}, {documents[j]!r}) = {score:.4f}")

Similarity('I love cats.', 'I like kittens.') = 0.7846
Similarity('I love cats.', 'Dogs are friendly animals.') = 0.3817
Similarity('I love cats.', 'The stock market is falling.') = 0.0394
Similarity('I love cats.', 'Python is a programming language.') = 0.1274
Similarity('I like kittens.', 'Dogs are friendly animals.') = 0.3099
Similarity('I like kittens.', 'The stock market is falling.') = 0.0402
Similarity('I like kittens.', 'Python is a programming language.') = 0.0927
Similarity('Dogs are friendly animals.', 'The stock market is falling.') = -0.0755
Similarity('Dogs are friendly animals.', 'Python is a programming language.') = 0.1034
Similarity('The stock market is falling.', 'Python is a programming language.') = 0.0697


In [ ]:
documents = [
    "I love cats.",
    "I like kittens.",
    "Dogs are friendly animals.",
    "The stock market is falling.",
    "Python is a programming language."
]

doc_embeddings = model.encode(documents)


In [ ]:
user_query = "I love animals"

In [ ]:
query_embedding = model.encode(user_query)

In [ ]:
# Compare query vector with every document vector, rank, return top-k
hits = util.semantic_search(query_embedding, doc_embeddings, top_k=3)

for hit in hits[0]:  # hits[0] because there's 1 query
    print(f"{documents[hit['corpus_id']]}  (score: {hit['score']:.4f})")

I love cats.  (score: 0.7603)
I like kittens.  (score: 0.6080)
Dogs are friendly animals.  (score: 0.5037)


In [ ]:
import numpy as np

np.random.seed(42)

num_vectors = 10_000
dimensions = 128

vectors = np.random.random((num_vectors, dimensions)).astype("float32")

query = np.random.random(dimensions).astype("float32")

In [ ]:
vectors.shape

(10000, 128)

In [ ]:
query.shape

(128,)

In [ ]:
import time

start = time.perf_counter()

# exact search:  Compare query vector with every document vector, rank, return top-k
hits = util.semantic_search(query, vectors, top_k=10)

for hit in hits[0]:  # hits[0] because there's 1 query
    print(f"{vectors[hit['corpus_id']]}  (score: {hit['score']:.4f})")

end = time.perf_counter()
latency = end - start
print(f"Latency of exact search: {latency}")

[0.68944246 0.26726052 0.62857044 0.6055053  0.14145759 0.9388708
 0.06197575 0.11327844 0.26651442 0.26792333 0.54374045 0.4336712
 0.7610008  0.7477615  0.4824787  0.532426   0.949551   0.81631935
 0.48347992 0.56451327 0.931513   0.5893412  0.9741865  0.6905204
 0.06484856 0.96412826 0.2328737  0.51615256 0.8084134  0.56714547
 0.09087437 0.07040505 0.9730381  0.3601048  0.8479005  0.83795875
 0.9062286  0.5449702  0.8230007  0.37724468 0.523677   0.39203238
 0.3500369  0.26476806 0.99643517 0.8006428  0.40575188 0.7322833
 0.5795836  0.5273749  0.69569457 0.26688293 0.97613937 0.09091342
 0.6343028  0.14713381 0.9405817  0.7468694  0.5298899  0.698997
 0.12609066 0.4509862  0.11187442 0.81802046 0.34912652 0.7073449
 0.5243885  0.7620702  0.8156387  0.6138118  0.89618254 0.8669871
 0.17482175 0.22982664 0.4264405  0.66770977 0.7957524  0.02535588
 0.7280545  0.82260185 0.30365312 0.840715   0.5233959  0.24505575
 0.7219315  0.8526273  0.61364174 0.45896888 0.7944376  0.8842141
 0.5

In [ ]:
indices = np.random.choice(num_vectors, size=2000, replace=False)
vectors_subset = vectors[indices]  # shape (1000, 128), float32

print(query.dtype, vectors_subset.dtype)  # should now print float32 float32

float32 float32


In [ ]:
vectors_subset.shape

(2000, 128)

In [ ]:
start = time.perf_counter()

# exact search:  Compare query vector with every document vector, rank, return top-k
hits = util.semantic_search(query, vectors_subset, top_k=10)

for hit in hits[0]:  # hits[0] because there's 1 query
    print(f"{vectors[hit['corpus_id']]}  (score: {hit['score']:.4f})")

end = time.perf_counter()
latency = end - start
print(f"Latency of approximate search: {latency}")

[0.49538535 0.83905834 0.40931177 0.3204335  0.49085215 0.37419242
 0.04396863 0.34019017 0.3291835  0.70647    0.17448232 0.06909481
 0.04807508 0.11170724 0.02255449 0.77082783 0.95446765 0.14433572
 0.09820356 0.6149749  0.3401001  0.60482347 0.72320855 0.9499517
 0.892104   0.10745724 0.04708683 0.0622225  0.6668856  0.44780073
 0.24442801 0.40019897 0.79418063 0.550303   0.7559052  0.89632547
 0.6538302  0.182896   0.19677892 0.26038846 0.08442821 0.87354714
 0.43348563 0.27147484 0.00160132 0.99670374 0.82844824 0.99117047
 0.18289469 0.0217128  0.37656674 0.26536772 0.9902233  0.03954423
 0.1066277  0.07086502 0.84114575 0.42377844 0.4986282  0.26441938
 0.13853134 0.69371647 0.9252003  0.8297515  0.7374656  0.3803515
 0.92847735 0.92627025 0.65240926 0.4002018  0.7173203  0.5594557
 0.34705564 0.551033   0.20021711 0.66814303 0.9091288  0.5303371
 0.321812   0.96291775 0.6373064  0.5205204  0.4056887  0.5477908
 0.49590522 0.8156483  0.8968987  0.78550726 0.32693642 0.61006236


In [ ]:
# Recall calculation

# recompute EXACT hits fresh (don't rely on old `hits` variable — it was overwritten)
exact_hits = util.semantic_search(query, vectors, top_k=10)
exact_ids = set(hit['corpus_id'] for hit in exact_hits[0])

# re-run approx search and capture hits
approx_hits = util.semantic_search(query, vectors_subset, top_k=10)

# map subset's local corpus_id back to original index in `vectors`
approx_ids = set(indices[hit['corpus_id']] for hit in approx_hits[0])

recall_at_10 = len(exact_ids & approx_ids) / 10
print(f"Recall@10: {recall_at_10:.2f}")

Recall@10: 0.30


In [ ]:
pip install hnswlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2807596 sha256=77a233c395818f5da69c0a20c2de51a36ef8f349581b76f42ad08396895d9ddf
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built hnswlib


In [ ]:
#HNSW EXPERIMENTS

import hnswlib

np.random.seed(42)

num_vectors = 10_000
dimensions = 128

vectors = np.random.random((num_vectors, dimensions)).astype("float32")
query = np.random.random(dimensions).astype("float32")

# 1. Create the HNSW index
index = hnswlib.Index(space="cosine", dim=dimensions)

# 2. Initialize it — M and ef_construction set here (build-time params)
index.init_index(max_elements=num_vectors, ef_construction=200, M=16)

# 3. Add the 10,000 vectors (IDs default to 0...num_vectors-1 if not provided)
ids = np.arange(num_vectors)
index.add_items(vectors, ids)

# 4. Set efSearch (query-time param)
index.set_ef(50)

# 5. Query for Top-10 nearest neighbors
labels, distances = index.knn_query(query, k=10)

print("Top-10 nearest neighbor IDs:", labels)
print("Their distances:", distances)

Top-10 nearest neighbor IDs: [[6267 9385 9571   82 8769 9352 4195 2825 5193 7018]]
Their distances: [[0.16189402 0.16906065 0.16919088 0.16976571 0.17068034 0.17567796
  0.17847931 0.18030351 0.18130606 0.18168604]]


In [ ]:
# Recall calculation

# recompute EXACT hits fresh (don't rely on old `hits` variable — it was overwritten)
exact_hits = util.semantic_search(query, vectors, top_k=10)
exact_ids = set(hit['corpus_id'] for hit in exact_hits[0])


In [ ]:
#REFACTOR THE CODE FOR BETTER UNDERSTANDING

In [ ]:
import numpy as np
import time
import hnswlib
from sentence_transformers import util

np.random.seed(42)

num_vectors = 10_000
dimensions = 128

vectors = np.random.random((num_vectors, dimensions)).astype("float32")
query = np.random.random(dimensions).astype("float32")

print(vectors.shape, query.shape)

(10000, 128) (128,)


In [ ]:
def exact_search(query, vectors, top_k=10):
    start = time.perf_counter()
    hits = util.semantic_search(query, vectors, top_k=top_k)[0]
    latency = time.perf_counter() - start
    ids = [hit['corpus_id'] for hit in hits]
    return ids, latency

exact_ids, exact_latency = exact_search(query, vectors, top_k=10)
print("Exact top-10 IDs:", exact_ids)
print(f"Exact latency: {exact_latency:.4f}s")

Exact top-10 IDs: [6267, 9385, 9571, 82, 8769, 9352, 4195, 3491, 3848, 2825]
Exact latency: 0.0065s


In [ ]:
def random_subset_search(query, vectors, subset_size=2000, top_k=10, seed=42):
    rng = np.random.default_rng(seed)
    indices = rng.choice(len(vectors), size=subset_size, replace=False)
    subset = vectors[indices]

    start = time.perf_counter()
    hits = util.semantic_search(query, subset, top_k=top_k)[0]
    latency = time.perf_counter() - start

    # map local subset IDs back to original vector IDs
    ids = [indices[hit['corpus_id']] for hit in hits]
    return ids, latency

subset_ids, subset_latency = random_subset_search(query, vectors, subset_size=2000, top_k=10)
print("Random-subset top-10 IDs:", subset_ids)
print(f"Random-subset latency: {subset_latency:.4f}s")

Random-subset top-10 IDs: [np.int64(9571), np.int64(82), np.int64(2825), np.int64(7018), np.int64(1536), np.int64(1056), np.int64(9525), np.int64(3054), np.int64(4613), np.int64(7142)]
Random-subset latency: 0.0089s


In [ ]:
# def build_hnsw_index(vectors, space="cosine", M=4, ef_construction=200):
#     dim = vectors.shape[1]
#     index = hnswlib.Index(space=space, dim=dim)
#     index.init_index(max_elements=len(vectors), ef_construction=ef_construction, M=M)
#     index.add_items(vectors, np.arange(len(vectors)))
#     return index

# def hnsw_search(index, query, ef_search=50, top_k=10):
#     index.set_ef(ef_search)
#     start = time.perf_counter()
#     labels, distances = index.knn_query(query, k=top_k)
#     latency = time.perf_counter() - start
#     return labels[0].tolist(), latency

# hnsw_index = build_hnsw_index(vectors, M=4, ef_construction=200)
# hnsw_ids, hnsw_latency = hnsw_search(hnsw_index, query, ef_search=50, top_k=10)
# print("HNSW top-10 IDs:", hnsw_ids)
# print(f"HNSW latency: {hnsw_latency:.4f}s")

In [ ]:
def build_hnsw_index(vectors, space="cosine", M=16, ef_construction=500):
    dim = vectors.shape[1]
    index = hnswlib.Index(space=space, dim=dim)
    index.init_index(max_elements=len(vectors), ef_construction=ef_construction, M=M)

    start = time.perf_counter()
    index.add_items(vectors, np.arange(len(vectors)))
    build_time = time.perf_counter() - start

    return index, build_time

def hnsw_search(index, query, ef_search=50, top_k=10):
    index.set_ef(ef_search)
    start = time.perf_counter()
    labels, distances = index.knn_query(query, k=top_k)
    latency = time.perf_counter() - start
    return labels[0].tolist(), latency

hnsw_index, build_time = build_hnsw_index(vectors, M=16, ef_construction=500)
hnsw_ids, hnsw_latency = hnsw_search(hnsw_index, query, ef_search=50, top_k=10)

print(f"Build time: {build_time:.4f}s")
print("HNSW top-10 IDs:", hnsw_ids)
print(f"Query latency: {hnsw_latency:.4f}s")

Build time: 4.2591s
HNSW top-10 IDs: [6267, 9385, 9571, 82, 8769, 9352, 4195, 2825, 5193, 7018]
Query latency: 0.0001s


In [ ]:
def recall_at_k(exact_ids, approx_ids, k=10):
    return len(set(exact_ids) & set(approx_ids)) / k

recall_subset = recall_at_k(exact_ids, subset_ids, k=10)
recall_hnsw = recall_at_k(exact_ids, hnsw_ids, k=10)

print(f"Recall@10 (random subset): {recall_subset:.2f}")
print(f"Recall@10 (HNSW):          {recall_hnsw:.2f}")

Recall@10 (random subset): 0.30
Recall@10 (HNSW):          0.80


In [ ]:
print(f"{'Method':<20}{'Latency (s)':<15}{'Recall@10':<10}")
print(f"{'Exact (brute force)':<20}{exact_latency:<15.4f}{'1.00':<10}")
print(f"{'Random subset':<20}{subset_latency:<15.4f}{recall_subset:<10.2f}")
print(f"{'HNSW':<20}{hnsw_latency:<15.4f}{recall_hnsw:<10.2f}")

Method              Latency (s)    Recall@10 
Exact (brute force) 0.0065         1.00      
Random subset       0.0089         0.30      
HNSW                0.0002         0.80      


In [ ]:
#BENCHMARKING FOR HNSW TO GET THE HIGH RECALL WITH LOW LATENCY
import pandas as pd

M_values = [4, 8, 16, 32]
ef_search_values = [10, 20, 50, 100, 200]
ef_construction = 200
top_k = 10

results = []

for M in M_values:
    # Build once per M — efSearch doesn't affect build
    hnsw_index, build_time = build_hnsw_index(vectors, M=M, ef_construction=ef_construction)

    for ef_search in ef_search_values:
        ids, query_latency = hnsw_search(hnsw_index, query, ef_search=ef_search, top_k=top_k)
        recall = recall_at_k(exact_ids, ids, k=top_k)

        results.append({
            "M": M,
            "efSearch": ef_search,
            "Build Time (s)": round(build_time, 4),
            "Search Latency (s)": round(query_latency, 6),
            "Recall@10": round(recall, 2)
        })

df = pd.DataFrame(results)
print(df.to_markdown(index=False))

|   M |   efSearch |   Build Time (s) |   Search Latency (s) |   Recall@10 |
|----:|-----------:|-----------------:|---------------------:|------------:|
|   4 |         10 |           1.2546 |             9e-05    |         0.2 |
|   4 |         20 |           1.2546 |             2.3e-05  |         0.2 |
|   4 |         50 |           1.2546 |             4.2e-05  |         0.2 |
|   4 |        100 |           1.2546 |             0.000102 |         0.5 |
|   4 |        200 |           1.2546 |             0.000202 |         0.6 |
|   8 |         10 |           2.2532 |             7.1e-05  |         0.4 |
|   8 |         20 |           2.2532 |             3.8e-05  |         0.4 |
|   8 |         50 |           2.2532 |             8e-05    |         0.5 |
|   8 |        100 |           2.2532 |             0.000148 |         0.8 |
|   8 |        200 |           2.2532 |             0.000288 |         0.8 |
|  16 |         10 |           3.4862 |             7.2e-05  |         0.4 |

In [ ]:
#BENCHMARKING FOR HNSW TO GET THE HIGH RECALL WITH LOW LATENCY WITH BRUTE FORCE LATENCY COLUMN
import pandas as pd

M_values = [4, 8, 16, 32]
ef_search_values = [10, 20, 50, 100, 200]
ef_construction = 200
top_k = 10

# Brute force baseline — computed once, same for every row
exact_ids, exact_latency = exact_search(query, vectors, top_k=top_k)

results = []

for M in M_values:
    # Build once per M — efSearch doesn't affect build
    hnsw_index, build_time = build_hnsw_index(vectors, M=M, ef_construction=ef_construction)

    for ef_search in ef_search_values:
        ids, query_latency = hnsw_search(hnsw_index, query, ef_search=ef_search, top_k=top_k)
        recall = recall_at_k(exact_ids, ids, k=top_k)

        results.append({
            "M": M,
            "efSearch": ef_search,
            "Build Time (s)": round(build_time, 4),
            "Search Latency (s)": round(query_latency, 6),
            "Brute Force Latency (s)": round(exact_latency, 6),
            "Speedup (x)": round(exact_latency / query_latency, 1) if query_latency > 0 else float("inf"),
            "Recall@10": round(recall, 2)
        })

df = pd.DataFrame(results)
print(df.to_markdown(index=False))

|   M |   efSearch |   Build Time (s) |   Search Latency (s) |   Brute Force Latency (s) |   Speedup (x) |   Recall@10 |
|----:|-----------:|-----------------:|---------------------:|--------------------------:|--------------:|------------:|
|   4 |         10 |           1.9692 |             0.000149 |                  0.036265 |         243   |         0.2 |
|   4 |         20 |           1.9692 |             4.4e-05  |                  0.036265 |         827.3 |         0.2 |
|   4 |         50 |           1.9692 |             6.9e-05  |                  0.036265 |         523.7 |         0.5 |
|   4 |        100 |           1.9692 |             0.000153 |                  0.036265 |         237.4 |         0.5 |
|   4 |        200 |           1.9692 |             0.000194 |                  0.036265 |         187.3 |         0.6 |
|   8 |         10 |           1.6833 |             5.5e-05  |                  0.036265 |         654.1 |         0.4 |
|   8 |         20 |           1